<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/04_control_flow_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · Control flow & functions

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: Python* - Instructor: Anderson Santos

## Learning objectives
- branch with **`if` / `elif` / `else`**;
- repeat with **`for`** (using `range`, `enumerate`, `zip`) and **`while`**;
- control loops with **`break`** / **`continue`** and catch errors with **`try`/`except`**;
- write **functions** with arguments, defaults, `*args`/`**kwargs`, docstrings;
- understand **scope** and **lambda**, and build a small sequence toolkit.

> **How to read this notebook while teaching.** Every code cell is commented: the lines starting with `#` explain what the code does and why it matters biologically, so you can narrate the class straight from the cell even without notes. Blockquotes marked **Instructor note** are extra talking points.

---
| Notebook | Topic |
|---|---|
| 01 | Python essentials & operators |
| 02 | Strings & biological sequences |
| 03 | Data structures |
| 04 | Control flow & functions |
| 05 | Files, scripting & modules |
| 06 | Pandas for omics |

## Setup — run this cell first

This cell makes the example data available whether you are on **your own computer** or on **Google Colab**. You do not need to understand it. Click it and press **Shift+Enter**.

> **Instructor note.** Set `GITHUB_REPO` below to your repository once, before sharing the notebooks.

In [1]:
# Run me first. Works on a local install AND on Google Colab.
import os

GITHUB_REPO = "andersonavilasantos/ufz-summerschool-python"   # <-- set to your GitHub repo

if not os.path.exists("../data/asv_table.csv"):
    # Data not found locally -> assume Google Colab and download the course files.
    os.system(f"git clone -q https://github.com/{GITHUB_REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")

assert os.path.exists("../data/asv_table.csv"), (
    "Could not find the data. On Colab, check GITHUB_REPO above; "
    "locally, run the notebook from inside the notebooks folder.")
print("✅ Setup complete — the data folder is available.")

✅ Setup complete — the data folder is available.


## 1. Conditionals — `if` / `elif` / `else`
Indentation (4 spaces) defines the block. No braces, no `end`.

In [ ]:
# 'if' runs a block only when a condition is True.
# INDENTATION (4 spaces) marks the block - Python has no { } or 'end'.
ph = 6.2
if ph < 6.5:
    reaction = 'acidic'      # runs if the first test is True
elif ph > 7.5:               # 'elif' = 'else if': checked only if above was False
    reaction = 'alkaline'
else:                        # runs if NOTHING above matched
    reaction = 'neutral'
print(f'pH {ph} is {reaction}')

## 2. `for` loops
Iterate directly over any collection. `range`, `enumerate` and `zip` are the three helpers you will reach for constantly.

In [ ]:
# A 'for' loop repeats its indented block once per item.
# You can loop DIRECTLY over a string's characters:
for base in 'ATGC':
    print(base, '->', base.replace('T', 'U'))   # transcribe each base

In [3]:
taxa = ['Pseudomonas', 'Bacillus', 'Nitrospira']
counts = [305, 120, 88]

# enumerate() gives you the INDEX and the item together:
for i, taxon in enumerate(taxa, start=1):   # start numbering at 1
    print(f'{i}. {taxon}')

# zip() walks TWO (or more) lists in parallel - taxon with its count:
for taxon, n in zip(taxa, counts):
    print(f'{taxon:12} {n}')

1. Pseudomonas
2. Bacillus
3. Nitrospira
Pseudomonas  305
Bacillus     120
Nitrospira   88


In [4]:
# range(start, stop, step) generates numbers - here, codon start positions.
seq = 'ATGGGCTTTAAC'
for i in range(0, len(seq) - 2, 3):   # 0, 3, 6, 9
    print(seq[i:i+3], end='  ')        # print each codon on one line

ATG  GGC  TTT  AAC  

## 3. `while`, `break`, `continue`
`while` repeats until a condition is false; `break` exits a loop early; `continue` skips to the next iteration.

In [5]:
# 'while' repeats AS LONG AS its condition stays True.
# Here: strip trailing low-quality 'N' bases from a read.
read = 'ATGCGGCNNN'
while read.endswith('N'):   # keep going until it no longer ends in N
    read = read[:-1]         # drop the last base
print('trimmed:', read)

trimmed: ATGCGGC


In [7]:
# 'break' leaves a loop immediately. Here: stop at the first stop codon.
seq = 'ATGGGCTAAGGGTTT'
for i in range(0, len(seq) - 2, 3):
    codon = seq[i:i+3]
    if codon in ('TAA', 'TAG', 'TGA'):   # the three stop codons
        print('stop codon at position', i)
        break                             # no need to scan further
# ('continue' would instead SKIP to the next codon without leaving the loop.)

stop codon at position 6


## 4. Catching errors — `try` / `except`
Real data is messy. `try/except` lets you handle bad values instead of crashing the whole analysis.

In [8]:
# Real files contain junk. try/except keeps one bad value from crashing everything.
raw = ['120', '305', 'NA', '88', '']   # 'NA' and '' are not numbers
clean = []
for value in raw:
    try:
        clean.append(int(value))    # TRY to convert to integer
    except ValueError:              # if that fails (e.g. 'NA') ...
        clean.append(0)             # ... use 0 instead of crashing
print('cleaned:', clean)

cleaned: [120, 305, 0, 88, 0]


## 5. Functions — write once, reuse everywhere
In notebooks 01–02 we computed GC content several times. A **function** captures it once. Use `def`, a **docstring**, and `return`.

In [ ]:
# 'def' DEFINES a function. The string right below is a DOCSTRING (its help).
def gc_content(seq):
    """Return the GC fraction (0-1) of a DNA sequence."""
    seq = seq.upper()                                    # normalise case
    return (seq.count('G') + seq.count('C')) / len(seq)  # 'return' hands back the result

print(gc_content('ATGCGC'))       # call it like any built-in function
print(f"{gc_content('AAAT'):.0%}")
help(gc_content)                    # help() prints the docstring

A function can **return several values** as a tuple:

In [ ]:
def composition(seq):
    """Return (length, gc_percent) for a sequence."""
    return len(seq), round(gc_content(seq) * 100, 1)   # two values -> a tuple

# ...which we can UNPACK straight into two names:
length, gc = composition('ATGCGCGC')
print('length =', length, ' GC% =', gc)

### Default, keyword, and variable arguments

In [ ]:
# Arguments can have DEFAULTS, used when the caller omits them.
def trim(seq, left=0, right=0):
    """Trim bases from each end; defaults trim nothing."""
    return seq[left: len(seq) - right]

print(trim('ATGCGCGT'))            # uses both defaults (no trimming)
print(trim('ATGCGCGT', left=2))    # name the argument you want to set
print(trim('ATGCGCGT', 1, 1))      # or pass by position: left=1, right=1

In [ ]:
# '*args' lets a function accept ANY number of arguments, bundled as a tuple.
def mean_length(*seqs):
    """Accept any number of sequences and average their lengths."""
    return sum(len(s) for s in seqs) / len(seqs)

print(mean_length('ATG', 'ATGCGC', 'ATGCGCGCG'))   # 3 args here, could be any number

In [ ]:
# '**kwargs' is the twin of *args: it bundles any number of NAMED arguments
# into a dictionary. Handy for optional, self-describing settings.
def describe_sample(sample_id, **fields):
    """Print a sample id plus any metadata passed by name."""
    print('sample:', sample_id)
    for key, value in fields.items():      # fields is a dict: {name: value}
        print(f'   {key} = {value}')

# Call it with whatever metadata you happen to have - the names are free:
describe_sample('S001', environment='Soil', pH=6.2, treatment='Amended')

### Scope
Variables created inside a function are **local** — they do not leak out.

In [ ]:
# Names created INSIDE a function are 'local' - invisible outside it.
def demo():
    inside = 42        # local variable
    return inside
print(demo())
# print(inside)       # <- would raise NameError: 'inside' is not defined out here

## 6. Lambda — tiny anonymous functions
Handy as a `key` for sorting.

In [ ]:
# A 'lambda' is a one-line function with no name - handy as a sort KEY.
seqs = ['ATGCGC', 'AT', 'ATGCGCGCGT', 'ATGC']
print('by length:', sorted(seqs, key=lambda s: len(s)))   # sort by length
print('by GC    :', sorted(seqs, key=gc_content, reverse=True))  # or by any function

## 7. A mini sequence toolkit
Combine functions into a reusable set. We will move these into a *module* in notebook 05.

In [ ]:
# Small, well-named functions COMPOSE into a toolkit you reuse everywhere.
def reverse_complement(seq):
    """Reverse complement of a DNA sequence."""
    return seq.upper().translate(str.maketrans('ACGT', 'TGCA'))[::-1]

def transcribe(seq):
    """DNA -> RNA."""
    return seq.upper().replace('T', 'U')

# Now use all three functions together on a couple of sequences:
for s in ['ATGGGC', 'TTTAAC']:
    print(f'{s}  rc={reverse_complement(s)}  rna={transcribe(s)}  gc={gc_content(s):.0%}')

---
## Exercises
**E1.** Write `at_content(seq)` returning the AT fraction (hint: `1 - gc_content(seq)`), and test it on `'ATGC'`.

**E2.** Loop over `['ATGC','GGGG','ATTT','GCGC']` and print, for each, whether it is `'GC-rich'` (GC ≥ 0.5) or `'AT-rich'`.

**E3.** Write `n_content(seq)` that returns the fraction of ambiguous `N` bases; test on `'ATGNNNCG'`.

**E4.** Given `raw = ['3.1','7.4','missing','6.8']` (pH values as text), use `try/except` to build a list of floats, skipping bad entries.

In [ ]:
# your code here

<details>
<summary><b>Solution: E1-E4</b> (click to expand)</summary>

```python
def at_content(seq):
    """Return AT fraction."""
    return 1 - gc_content(seq)
print(at_content('ATGC'))

for s in ['ATGC','GGGG','ATTT','GCGC']:
    print(s, 'GC-rich' if gc_content(s) >= 0.5 else 'AT-rich')

def n_content(seq):
    return seq.upper().count('N') / len(seq)
print(n_content('ATGNNNCG'))

raw = ['3.1','7.4','missing','6.8']
vals = []
for x in raw:
    try: vals.append(float(x))
    except ValueError: pass
print(vals)
```
</details>

### Recap
- branch with `if/elif/else`; loop with `for` (`range/enumerate/zip`) and `while`.
- `break`/`continue` steer loops; `try/except` survives messy data.
- functions: `def`, docstring, `return`, defaults, `*args`, local scope, `lambda`.

Next: **05 · Files, scripting & modules**.